# 11 — Thermal-sensitivity falsification test (Paso 3, user protocol 2026-09-11)

Re-runs the actual O3 equation discovery on the same selected no-lag feature set
(`photochemical_potential, no, pm25, rh_2m, w700, day_of_week, no2, precip_total_mm`,
the config that gives LOYO R2=0.640 in the main pipeline) but adds `temp_2m` as an
additional candidate variable in the PySR search pool.

- If PySR replaces the `pm25`/`photochemical_potential`-driven interaction with a
  `temp_2m*radiation`-shaped term, PM2.5 was a statistical stand-in for thermal energy
  (matching the temperature-tercile result showing the interaction strengthens on warm
  days -- Hipotesis B/C, evaporative-VOC or direct-thermal pathways).
- If PySR keeps a real PM2.5-involving term ALONGSIDE a temp_2m term on the Pareto
  front, secondary particulate mass and temperature exert independent predictive
  pressure on O3.


In [1]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

DATA_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / "bogota"

df = pd.read_parquet(DATA_DIR / "bogota_o3_master_dataset.parquet")
df = df.dropna(subset=['o3']).reset_index(drop=True)

with open(DATA_DIR / "bogota_o3_feature_selection.json") as f:
    selection = json.load(f)
base_features = selection['selected_no_lag']
features = base_features + ['temp_2m', 'pm25']   # 2026-10-20: PM2.5 and temperature both offered (the corrected-fire selection no longer picks PM2.5)
print("Base (existing R2=0.640) features:", base_features)
print("Features with temp_2m added:", features)

data = df.dropna(subset=features + ['o3']).reset_index(drop=True)
X = data[features].to_numpy()
y = data['o3'].to_numpy()
print(data.shape)


Base (existing R2=0.640) features: ['photochemical_potential', 'hotspots_sector_E_lag2', 'fog_stagnation', 'wind_speed_era5', 'no_ratio', 'w700', 'radiation_mean']
Features with temp_2m added: ['photochemical_potential', 'hotspots_sector_E_lag2', 'fog_stagnation', 'wind_speed_era5', 'no_ratio', 'w700', 'radiation_mean', 'temp_2m', 'pm25']
(2917, 219)


## PySR with temp_2m as a candidate alongside the existing selected features

In [2]:
from pysr import PySRRegressor

sr = PySRRegressor(
    niterations=200,
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["log", "sqrt", "exp"],
    model_selection="best",
    maxsize=25,
    populations=24,
)
sr.fit(X, y, variable_names=features)
print(sr)


Compiling Julia backend...


[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 1.140e+04
Progress: 28 / 200 total iterations (14.354%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.894e+01  0.000e+00  y = 20.123
3           4.386e+01  5.483e-02  y = no_ratio - -18.882
4           3.692e+01  1.723e-01  y = sqrt(photochemical_potential * 0.10044)
6           2.839e+01  1.313e-01  y = sqrt(radiation_mean + (no_ratio * radiation_mean))
7           2.828e+01  3.950e-03  y = sqrt((radiation_mean + radiation_mean) * sqrt(no_ratio...
                                      ))
8           2.243e+01  2.316e-01  y = sqrt(no_ratio * radiation_mean) - (pm25 * -0.31826)
9           2.242e+01  6.560e-04  y = sqrt((radiation_mean * no_ratio) + sqrt(radiation_mean...
                                       * hotspots_sector_E_lag2))
10     

PySRRegressor.equations_ = [
	    pick     score                                           equation  \
	0         0.000000                                          20.142508   
	1         0.083494                        radiation_mean * 0.10520633   
	2         0.115210         sqrt(photochemical_potential) * 0.31869584   
	3         0.055273                          temp_2m * log(sqrt(pm25))   
	4         0.207057  sqrt(radiation_mean + (no_ratio * radiation_me...   
	5         0.110012               temp_2m * log(sqrt(pm25 * no_ratio))   
	6   >>>>  0.196083  sqrt(radiation_mean) * log(sqrt(pm25 * no_ratio))   
	7         0.017111  sqrt(radiation_mean * sqrt(pm25 * no_ratio)) *...   
	8         0.020333  sqrt(radiation_mean) * log(sqrt(wind_speed_era...   
	9         0.019813  (temp_2m * sqrt(sqrt(no_ratio * pm25) * radiat...   
	10        0.034525  sqrt((sqrt(pm25 * no_ratio) * 0.029881364) * (...   
	11        0.019542  sqrt((hotspots_sector_E_lag2 + (sqrt(no_ratio)...   
	12      

In [3]:
pareto = sr.equations_[['complexity', 'loss', 'equation']]
pareto['r2'] = pareto.apply(lambda row: r2_score(y, sr.predict(X, index=row.name)), axis=1)
pareto


,complexity,loss,equation,r2
0,1,48.941936,20.142508,-1.601566e-07
1,3,41.415180,radiation_mean * 0.10520633,1.537893e-01
2,4,36.908320,sqrt(photochemical_potential) * 0.31869584,2.458752e-01
3,5,34.923640,temp_2m * log(sqrt(pm25)),2.864269e-01
4,6,28.391994,sqrt(radiation_mean + (no_ratio * radiation_me...,4.198840e-01
5,7,25.434200,temp_2m * log(sqrt(pm25 * no_ratio)),4.803188e-01
6,8,20.905497,sqrt(radiation_mean) * log(sqrt(pm25 * no_ratio)),5.728509e-01
7,9,20.550833,sqrt(radiation_mean * sqrt(pm25 * no_ratio)) *...,5.800976e-01
8,10,20.137184,sqrt(radiation_mean) * log(sqrt(wind_speed_era...,5.885494e-01
9,11,19.742126,(temp_2m * sqrt(sqrt(no_ratio * pm25) * radiat...,5.966213e-01


## Interpretation: does temp_2m displace pm25, or coexist with it?

Check the highest-R2 equations for whether `pm25` and `temp_2m` both appear, only one
does, or neither in favor of some other combination.


In [4]:
print("Full Pareto front:")
for _, row in pareto.sort_values('complexity').iterrows():
    has_pm25 = 'pm25' in row['equation']
    has_temp = 'temp_2m' in row['equation']
    print(f"  [{int(row['complexity']):2d}] R2={row['r2']:.3f}  pm25={has_pm25}  temp_2m={has_temp}  {row['equation']}")

print()
print("Best (highest-R2) equation:")
best = pareto.loc[pareto['r2'].idxmax()]
print(best['equation'])
print(f"R2 = {best['r2']:.3f}  (vs. existing no-temp_2m best in-sample R2, for comparison)")


Full Pareto front:
  [ 1] R2=-0.000  pm25=False  temp_2m=False  20.142508
  [ 3] R2=0.154  pm25=False  temp_2m=False  radiation_mean * 0.10520633
  [ 4] R2=0.246  pm25=False  temp_2m=False  sqrt(photochemical_potential) * 0.31869584
  [ 5] R2=0.286  pm25=True  temp_2m=True  temp_2m * log(sqrt(pm25))
  [ 6] R2=0.420  pm25=False  temp_2m=False  sqrt(radiation_mean + (no_ratio * radiation_mean))
  [ 7] R2=0.480  pm25=True  temp_2m=True  temp_2m * log(sqrt(pm25 * no_ratio))
  [ 8] R2=0.573  pm25=True  temp_2m=False  sqrt(radiation_mean) * log(sqrt(pm25 * no_ratio))
  [ 9] R2=0.580  pm25=True  temp_2m=False  sqrt(radiation_mean * sqrt(pm25 * no_ratio)) * 0.70963
  [10] R2=0.589  pm25=True  temp_2m=False  sqrt(radiation_mean) * log(sqrt(wind_speed_era5 + (no_ratio * pm25)))
  [11] R2=0.597  pm25=True  temp_2m=True  (temp_2m * sqrt(sqrt(no_ratio * pm25) * radiation_mean)) * 0.047909565
  [13] R2=0.624  pm25=True  temp_2m=True  sqrt((sqrt(pm25 * no_ratio) * 0.029881364) * (hotspots_sector_E_la

## Save

In [5]:
pareto.to_csv(DATA_DIR / "bogota_o3_temp_pm_falsification_pareto.csv", index=False)
print("Saved")


Saved
